# Model Drift Monitoring and Stress Testing

## Aim

This notebook develops a reproducible monitoring framework for the customer-churn project.

The analysis has three purposes:

1. measure whether feature distributions changed between the training, validation and test datasets;
2. monitor whether model predictions and performance remained stable across the validation and test periods;
3. simulate a realistic deterioration in customer behaviour and test how the selected manual model responds.

The selected Random Forest pipeline is used for the executable stress test because its complete preprocessing and classification pipeline was saved locally. The final Azure AutoML test probabilities are also included as contextual evidence, but AutoML is not rescored on the synthetically drifted features in this local notebook.

This distinction is reported transparently. In deployment, the same feature and prediction-monitoring rules would be applied to probabilities returned by the AutoML endpoint.

## Main outputs

The notebook creates:

- feature-level Population Stability Index results;
- numeric and categorical drift summaries;
- missingness changes;
- validation-versus-test prediction drift;
- simulated-drift performance results;
- operational drift alerts;
- figures and a dissertation-ready conclusion.

## Methodological approach

The training set is treated as the reference distribution.

Observed feature drift is assessed by comparing:

- training versus validation;
- training versus test.

Population Stability Index (PSI) is interpreted using commonly applied operational thresholds:

- **PSI below 0.10:** stable;
- **PSI from 0.10 to below 0.25:** warning;
- **PSI of 0.25 or above:** significant drift.

For numeric variables, the reference distribution defines quantile-based bins. For categorical variables, category proportions are compared directly.

The simulated stress test changes selected behavioural variables in a direction that may represent declining engagement, reduced renewal activity and longer inactivity. The original class labels are retained so that the effect on model discrimination and classification performance can be measured.

## 1. Import libraries and configure file paths

The default paths match the folders used in the completed dissertation workflow.

Change a path only when the file is stored elsewhere.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)


RANDOM_STATE = 42
TARGET_COLUMN = "is_churn"
CLASSIFICATION_THRESHOLD = 0.50

TRAIN_PATH = Path(
    r"E:\customer_retention_dissertation_outputs\customer_retention_train.csv"
)

VALIDATION_PATH = Path(
    r"E:\customer_retention_dissertation_outputs\customer_retention_validation.csv"
)

TEST_PATH = Path(
    r"E:\customer_retention_dissertation_outputs\customer_retention_test.csv"
)

MANUAL_MODEL_PATH = Path(
    r"E:\manual_automl_comparison_outputs\07_selected_manual_pipeline_training_only.joblib"
)

FINAL_PREDICTION_EVIDENCE_PATH = Path(
    r"E:\manual_automl_comparison_outputs\12_final_test_prediction_evidence.csv"
)

OUTPUT_DIR = (
    Path.cwd()
    / "model_drift_monitoring_outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Output folder:", OUTPUT_DIR.resolve())

## 2. Confirm the required files

This check prevents the analysis from continuing with an incorrect or incomplete path.

In [ ]:
required_files = {
    "training data": TRAIN_PATH,
    "validation data": VALIDATION_PATH,
    "test data": TEST_PATH,
    "manual model": MANUAL_MODEL_PATH,
}

file_check_rows = []

for file_name, file_path in (
    required_files.items()
):
    file_check_rows.append(
        {
            "required_item": file_name,
            "path": str(file_path),
            "exists": file_path.exists(),
        }
    )

file_check = pd.DataFrame(
    file_check_rows
)

display(
    file_check
)

missing_files = file_check.loc[
    ~file_check["exists"],
    "required_item",
].tolist()

if missing_files:
    raise FileNotFoundError(
        "Missing required file(s): "
        + ", ".join(missing_files)
        + ". Correct the paths in Section 1."
    )

print(
    "Required file check passed."
)

## 3. Load the datasets and saved model

The three processed datasets should use the same predictor schema and should contain the binary target `is_churn`.

The saved Random Forest object should be the full preprocessing and classification pipeline selected in the manual modelling stage.

In [ ]:
train_df = pd.read_csv(
    TRAIN_PATH,
    low_memory=False,
)

validation_df = pd.read_csv(
    VALIDATION_PATH,
    low_memory=False,
)

test_df = pd.read_csv(
    TEST_PATH,
    low_memory=False,
)

selected_manual_pipeline = (
    joblib.load(
        MANUAL_MODEL_PATH
    )
)

print(
    "Training shape:",
    train_df.shape,
)

print(
    "Validation shape:",
    validation_df.shape,
)

print(
    "Test shape:",
    test_df.shape,
)

print(
    "Loaded model type:",
    type(
        selected_manual_pipeline
    ),
)

## 4. Validate the schema and prepare predictors

Customer identifiers are excluded from the model features. The code then checks that the predictor columns are identical across the three data partitions.

In [ ]:
identifier_candidates = {
    "msno",
    "customer_id",
    "customerid",
    "member_id",
    "memberid",
    "user_id",
    "userid",
}


def prepare_split(
    frame,
    split_name,
):
    if TARGET_COLUMN not in frame.columns:
        raise KeyError(
            f"{TARGET_COLUMN} is missing "
            f"from the {split_name} dataset."
        )

    target = pd.to_numeric(
        frame[
            TARGET_COLUMN
        ],
        errors="raise",
    ).astype(int)

    predictors = frame.drop(
        columns=[
            TARGET_COLUMN
        ]
    ).copy()

    identifier_columns = [
        column
        for column in predictors.columns
        if column.lower()
        in identifier_candidates
    ]

    if identifier_columns:
        predictors = predictors.drop(
            columns=identifier_columns
        )

    return predictors, target


X_train, y_train = prepare_split(
    train_df,
    "training",
)

X_validation, y_validation = (
    prepare_split(
        validation_df,
        "validation",
    )
)

X_test, y_test = prepare_split(
    test_df,
    "test",
)


training_columns = list(
    X_train.columns
)

validation_columns = list(
    X_validation.columns
)

test_columns = list(
    X_test.columns
)

if (
    training_columns
    != validation_columns
    or training_columns
    != test_columns
):
    raise ValueError(
        "The predictor columns or column order "
        "differ between the data splits."
    )

numeric_features = (
    X_train.select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
    .tolist()
)

categorical_features = [
    column
    for column in X_train.columns
    if column not in numeric_features
]

print(
    "Predictors:",
    len(training_columns),
)

print(
    "Numeric features:",
    len(numeric_features),
)

print(
    "Categorical features:",
    len(categorical_features),
)

print(
    "Training churn rate:",
    f"{y_train.mean():.4f}",
)

print(
    "Validation churn rate:",
    f"{y_validation.mean():.4f}",
)

print(
    "Test churn rate:",
    f"{y_test.mean():.4f}",
)

## 5. Define drift and performance functions

The functions below calculate:

- numeric PSI using quantile bins from the reference data;
- categorical PSI using observed category proportions;
- Kolmogorov–Smirnov statistics for numeric features;
- total variation distance for categorical features;
- model performance metrics;
- probability-distribution PSI.

In [ ]:
PSI_EPSILON = 1e-6


def classify_psi(
    psi_value,
):
    if pd.isna(
        psi_value
    ):
        return "not_available"

    if psi_value < 0.10:
        return "stable"

    if psi_value < 0.25:
        return "warning"

    return "significant"


def calculate_psi_from_proportions(
    reference_proportions,
    current_proportions,
):
    reference_proportions = np.asarray(
        reference_proportions,
        dtype=float,
    )

    current_proportions = np.asarray(
        current_proportions,
        dtype=float,
    )

    reference_proportions = np.clip(
        reference_proportions,
        PSI_EPSILON,
        None,
    )

    current_proportions = np.clip(
        current_proportions,
        PSI_EPSILON,
        None,
    )

    return float(
        np.sum(
            (
                current_proportions
                - reference_proportions
            )
            * np.log(
                current_proportions
                / reference_proportions
            )
        )
    )


def numeric_psi(
    reference_series,
    current_series,
    number_of_bins=10,
):
    reference = pd.to_numeric(
        reference_series,
        errors="coerce",
    ).dropna()

    current = pd.to_numeric(
        current_series,
        errors="coerce",
    ).dropna()

    if (
        reference.empty
        or current.empty
    ):
        return np.nan

    if reference.nunique() <= 1:
        return (
            0.0
            if current.nunique() <= 1
            and current.iloc[0]
            == reference.iloc[0]
            else np.nan
        )

    quantiles = np.linspace(
        0,
        1,
        number_of_bins + 1,
    )

    bin_edges = np.unique(
        reference.quantile(
            quantiles
        ).to_numpy()
    )

    if len(bin_edges) < 3:
        minimum = float(
            reference.min()
        )

        maximum = float(
            reference.max()
        )

        if minimum == maximum:
            return np.nan

        bin_edges = np.linspace(
            minimum,
            maximum,
            number_of_bins + 1,
        )

    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    reference_bins = pd.cut(
        reference,
        bins=bin_edges,
        include_lowest=True,
    )

    current_bins = pd.cut(
        current,
        bins=bin_edges,
        include_lowest=True,
    )

    reference_proportions = (
        reference_bins
        .value_counts(
            normalize=True,
            sort=False,
        )
        .to_numpy()
    )

    current_proportions = (
        current_bins
        .value_counts(
            normalize=True,
            sort=False,
        )
        .reindex(
            reference_bins.cat.categories,
            fill_value=0,
        )
        .to_numpy()
    )

    return (
        calculate_psi_from_proportions(
            reference_proportions,
            current_proportions,
        )
    )


def categorical_psi(
    reference_series,
    current_series,
):
    reference = (
        reference_series
        .astype("string")
        .fillna("__MISSING__")
    )

    current = (
        current_series
        .astype("string")
        .fillna("__MISSING__")
    )

    categories = sorted(
        set(reference.unique())
        | set(current.unique())
    )

    reference_proportions = (
        reference.value_counts(
            normalize=True
        )
        .reindex(
            categories,
            fill_value=0,
        )
        .to_numpy()
    )

    current_proportions = (
        current.value_counts(
            normalize=True
        )
        .reindex(
            categories,
            fill_value=0,
        )
        .to_numpy()
    )

    return (
        calculate_psi_from_proportions(
            reference_proportions,
            current_proportions,
        )
    )


def categorical_total_variation(
    reference_series,
    current_series,
):
    reference = (
        reference_series
        .astype("string")
        .fillna("__MISSING__")
    )

    current = (
        current_series
        .astype("string")
        .fillna("__MISSING__")
    )

    categories = sorted(
        set(reference.unique())
        | set(current.unique())
    )

    reference_proportions = (
        reference.value_counts(
            normalize=True
        )
        .reindex(
            categories,
            fill_value=0,
        )
        .to_numpy()
    )

    current_proportions = (
        current.value_counts(
            normalize=True
        )
        .reindex(
            categories,
            fill_value=0,
        )
        .to_numpy()
    )

    return float(
        0.5
        * np.abs(
            reference_proportions
            - current_proportions
        ).sum()
    )


def calculate_model_metrics(
    actual,
    probability,
    threshold=0.50,
):
    actual = np.asarray(
        actual,
        dtype=int,
    )

    probability = np.asarray(
        probability,
        dtype=float,
    )

    prediction = (
        probability
        >= threshold
    ).astype(int)

    return {
        "accuracy": accuracy_score(
            actual,
            prediction,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                actual,
                prediction,
            )
        ),
        "precision_churn": (
            precision_score(
                actual,
                prediction,
                zero_division=0,
            )
        ),
        "recall_churn": (
            recall_score(
                actual,
                prediction,
                zero_division=0,
            )
        ),
        "f1_churn": f1_score(
            actual,
            prediction,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            actual,
            probability,
        ),
        "average_precision": (
            average_precision_score(
                actual,
                probability,
            )
        ),
        "log_loss": log_loss(
            actual,
            probability,
            labels=[
                0,
                1,
            ],
        ),
        "mean_probability": float(
            probability.mean()
        ),
        "predicted_churn_rate": float(
            prediction.mean()
        ),
    }

## 6. Measure observed feature drift

Each feature is compared with the training reference distribution.

Numeric variables include PSI and a Kolmogorov–Smirnov statistic. Categorical variables include PSI and total variation distance.

In [ ]:
observed_comparisons = {
    "training_vs_validation": (
        X_validation
    ),
    "training_vs_test": (
        X_test
    ),
}

feature_drift_rows = []

for comparison_name, current_frame in (
    observed_comparisons.items()
):
    for feature in numeric_features:
        psi_value = numeric_psi(
            X_train[feature],
            current_frame[
                feature
            ],
        )

        reference_numeric = (
            pd.to_numeric(
                X_train[
                    feature
                ],
                errors="coerce",
            )
            .dropna()
        )

        current_numeric = (
            pd.to_numeric(
                current_frame[
                    feature
                ],
                errors="coerce",
            )
            .dropna()
        )

        if (
            reference_numeric.empty
            or current_numeric.empty
        ):
            ks_statistic = np.nan
            ks_p_value = np.nan
        else:
            ks_result = ks_2samp(
                reference_numeric,
                current_numeric,
                alternative="two-sided",
                mode="auto",
            )

            ks_statistic = float(
                ks_result.statistic
            )

            ks_p_value = float(
                ks_result.pvalue
            )

        feature_drift_rows.append(
            {
                "comparison": (
                    comparison_name
                ),
                "feature": feature,
                "feature_type": (
                    "numeric"
                ),
                "psi": psi_value,
                "drift_severity": (
                    classify_psi(
                        psi_value
                    )
                ),
                "ks_statistic": (
                    ks_statistic
                ),
                "ks_p_value": (
                    ks_p_value
                ),
                "total_variation_distance": (
                    np.nan
                ),
                "reference_missing_rate": (
                    X_train[
                        feature
                    ].isna().mean()
                ),
                "current_missing_rate": (
                    current_frame[
                        feature
                    ].isna().mean()
                ),
            }
        )

    for feature in categorical_features:
        psi_value = categorical_psi(
            X_train[feature],
            current_frame[
                feature
            ],
        )

        total_variation = (
            categorical_total_variation(
                X_train[
                    feature
                ],
                current_frame[
                    feature
                ],
            )
        )

        feature_drift_rows.append(
            {
                "comparison": (
                    comparison_name
                ),
                "feature": feature,
                "feature_type": (
                    "categorical"
                ),
                "psi": psi_value,
                "drift_severity": (
                    classify_psi(
                        psi_value
                    )
                ),
                "ks_statistic": (
                    np.nan
                ),
                "ks_p_value": (
                    np.nan
                ),
                "total_variation_distance": (
                    total_variation
                ),
                "reference_missing_rate": (
                    X_train[
                        feature
                    ].isna().mean()
                ),
                "current_missing_rate": (
                    current_frame[
                        feature
                    ].isna().mean()
                ),
            }
        )

feature_drift_summary = pd.DataFrame(
    feature_drift_rows
)

feature_drift_summary[
    "missing_rate_change"
] = (
    feature_drift_summary[
        "current_missing_rate"
    ]
    - feature_drift_summary[
        "reference_missing_rate"
    ]
)

feature_drift_summary.to_csv(
    OUTPUT_DIR
    / "01_feature_drift_summary.csv",
    index=False,
)

numeric_psi_results = (
    feature_drift_summary.loc[
        feature_drift_summary[
            "feature_type"
        ]
        == "numeric"
    ]
    .copy()
)

numeric_psi_results.to_csv(
    OUTPUT_DIR
    / "02_numeric_psi_results.csv",
    index=False,
)

categorical_drift_results = (
    feature_drift_summary.loc[
        feature_drift_summary[
            "feature_type"
        ]
        == "categorical"
    ]
    .copy()
)

categorical_drift_results.to_csv(
    OUTPUT_DIR
    / "03_categorical_drift_results.csv",
    index=False,
)

display(
    feature_drift_summary.sort_values(
        [
            "comparison",
            "psi",
        ],
        ascending=[
            True,
            False,
        ],
    ).head(30)
)

## 7. Score validation and test data with the saved manual pipeline

This establishes whether the model's output distribution and predictive performance remained stable before the synthetic stress test.

In [ ]:
validation_probability = (
    selected_manual_pipeline
    .predict_proba(
        X_validation
    )[:, 1]
)

test_probability = (
    selected_manual_pipeline
    .predict_proba(
        X_test
    )[:, 1]
)

validation_metrics = (
    calculate_model_metrics(
        actual=y_validation,
        probability=(
            validation_probability
        ),
        threshold=(
            CLASSIFICATION_THRESHOLD
        ),
    )
)

test_metrics = (
    calculate_model_metrics(
        actual=y_test,
        probability=test_probability,
        threshold=(
            CLASSIFICATION_THRESHOLD
        ),
    )
)

observed_performance = pd.DataFrame(
    [
        {
            "dataset": "validation",
            **validation_metrics,
        },
        {
            "dataset": "test",
            **test_metrics,
        },
    ]
)

display(
    observed_performance.style.format(
        {
            "accuracy": "{:.4f}",
            "balanced_accuracy": "{:.4f}",
            "precision_churn": "{:.4f}",
            "recall_churn": "{:.4f}",
            "f1_churn": "{:.4f}",
            "roc_auc": "{:.4f}",
            "average_precision": "{:.4f}",
            "log_loss": "{:.4f}",
            "mean_probability": "{:.4f}",
            "predicted_churn_rate": (
                "{:.4f}"
            ),
        }
    )
)

## 8. Measure observed prediction drift

The validation probability distribution is used as the reference and the test distribution is treated as the current period.

The prediction PSI measures whether the model's output distribution changed even when overall performance remained similar.

In [ ]:
observed_prediction_psi = (
    numeric_psi(
        pd.Series(
            validation_probability
        ),
        pd.Series(
            test_probability
        ),
    )
)

prediction_drift_summary = (
    pd.DataFrame(
        [
            {
                "comparison": (
                    "validation_vs_test"
                ),
                "model": (
                    "Random Forest"
                ),
                "probability_psi": (
                    observed_prediction_psi
                ),
                "prediction_drift_severity": (
                    classify_psi(
                        observed_prediction_psi
                    )
                ),
                "reference_mean_probability": (
                    float(
                        validation_probability.mean()
                    )
                ),
                "current_mean_probability": (
                    float(
                        test_probability.mean()
                    )
                ),
                "mean_probability_change": (
                    float(
                        test_probability.mean()
                        - validation_probability.mean()
                    )
                ),
                "reference_predicted_churn_rate": (
                    validation_metrics[
                        "predicted_churn_rate"
                    ]
                ),
                "current_predicted_churn_rate": (
                    test_metrics[
                        "predicted_churn_rate"
                    ]
                ),
                "predicted_churn_rate_change": (
                    test_metrics[
                        "predicted_churn_rate"
                    ]
                    - validation_metrics[
                        "predicted_churn_rate"
                    ]
                ),
            }
        ]
    )
)

display(
    prediction_drift_summary
)

## 9. Create a realistic simulated drift scenario

The stress test applies deterministic changes to available features associated with customer activity, transaction behaviour, renewal and pricing.

The changes are illustrative rather than forecasts. They represent a period in which customers are less active, renew less frequently and show weaker transaction engagement.

The code changes only features that exist in the dataset, so it remains robust to small schema differences.

In [ ]:
rng = np.random.default_rng(
    RANDOM_STATE
)

X_test_drifted = X_test.copy()

applied_drift_rows = []


def apply_numeric_multiplier(
    frame,
    feature,
    multiplier,
    minimum_value=None,
    maximum_value=None,
):
    if feature not in frame.columns:
        return

    original_values = pd.to_numeric(
        frame[feature],
        errors="coerce",
    )

    drifted_values = (
        original_values
        * multiplier
    )

    if minimum_value is not None:
        drifted_values = (
            drifted_values.clip(
                lower=minimum_value
            )
        )

    if maximum_value is not None:
        drifted_values = (
            drifted_values.clip(
                upper=maximum_value
            )
        )

    frame[feature] = drifted_values

    applied_drift_rows.append(
        {
            "feature": feature,
            "operation": (
                "numeric_multiplier"
            ),
            "parameter": multiplier,
            "original_mean": float(
                original_values.mean()
            ),
            "drifted_mean": float(
                drifted_values.mean()
            ),
        }
    )


apply_numeric_multiplier(
    X_test_drifted,
    "tx_count",
    0.75,
    minimum_value=0,
)

apply_numeric_multiplier(
    X_test_drifted,
    "transaction_history_days",
    0.85,
    minimum_value=0,
)

apply_numeric_multiplier(
    X_test_drifted,
    "days_since_last_transaction",
    1.35,
    minimum_value=0,
)

apply_numeric_multiplier(
    X_test_drifted,
    "is_auto_renew_mean",
    0.70,
    minimum_value=0,
    maximum_value=1,
)

apply_numeric_multiplier(
    X_test_drifted,
    "days_since_last_activity",
    1.30,
    minimum_value=0,
)

apply_numeric_multiplier(
    X_test_drifted,
    "activity_history_days",
    0.90,
    minimum_value=0,
)

apply_numeric_multiplier(
    X_test_drifted,
    "log_count",
    0.80,
    minimum_value=0,
)

apply_numeric_multiplier(
    X_test_drifted,
    "plan_list_price_mean",
    1.10,
    minimum_value=0,
)

apply_numeric_multiplier(
    X_test_drifted,
    "actual_amount_paid_mean",
    1.08,
    minimum_value=0,
)


payment_feature = (
    "latest_payment_method"
)

if payment_feature in (
    X_test_drifted.columns
):
    payment_values = (
        X_test_drifted[
            payment_feature
        ]
        .astype("string")
    )

    payment_counts = (
        payment_values
        .value_counts(
            dropna=True
        )
    )

    if len(payment_counts) >= 2:
        dominant_category = (
            payment_counts.index[0]
        )

        eligible_indices = (
            payment_values.loc[
                payment_values
                != dominant_category
            ]
            .index
            .to_numpy()
        )

        number_to_shift = int(
            0.25
            * len(
                eligible_indices
            )
        )

        if number_to_shift > 0:
            selected_indices = (
                rng.choice(
                    eligible_indices,
                    size=(
                        number_to_shift
                    ),
                    replace=False,
                )
            )

            X_test_drifted.loc[
                selected_indices,
                payment_feature,
            ] = dominant_category

            applied_drift_rows.append(
                {
                    "feature": (
                        payment_feature
                    ),
                    "operation": (
                        "categorical_concentration"
                    ),
                    "parameter": (
                        "25% of non-dominant "
                        "categories moved to "
                        "the dominant category"
                    ),
                    "original_mean": (
                        np.nan
                    ),
                    "drifted_mean": (
                        np.nan
                    ),
                }
            )

applied_drift_summary = pd.DataFrame(
    applied_drift_rows
)

applied_drift_summary.to_csv(
    OUTPUT_DIR
    / "04_simulated_drift_definition.csv",
    index=False,
)

display(
    applied_drift_summary
)

## 10. Quantify the simulated feature drift

The original test data is treated as the reference period and the synthetically changed test data is treated as the drifted period.

In [ ]:
simulated_feature_rows = []

for feature in numeric_features:
    psi_value = numeric_psi(
        X_test[feature],
        X_test_drifted[
            feature
        ],
    )

    simulated_feature_rows.append(
        {
            "comparison": (
                "original_test_vs_"
                "simulated_drift"
            ),
            "feature": feature,
            "feature_type": (
                "numeric"
            ),
            "psi": psi_value,
            "drift_severity": (
                classify_psi(
                    psi_value
                )
            ),
        }
    )

for feature in categorical_features:
    psi_value = categorical_psi(
        X_test[feature],
        X_test_drifted[
            feature
        ],
    )

    simulated_feature_rows.append(
        {
            "comparison": (
                "original_test_vs_"
                "simulated_drift"
            ),
            "feature": feature,
            "feature_type": (
                "categorical"
            ),
            "psi": psi_value,
            "drift_severity": (
                classify_psi(
                    psi_value
                )
            ),
        }
    )

simulated_feature_drift = (
    pd.DataFrame(
        simulated_feature_rows
    )
)

simulated_feature_drift.to_csv(
    OUTPUT_DIR
    / "05_simulated_feature_drift.csv",
    index=False,
)

display(
    simulated_feature_drift.sort_values(
        "psi",
        ascending=False,
    ).head(20)
)

## 11. Score the simulated drift data

The same saved model and unchanged class labels are used. This isolates the effect of covariate change on the model's predictions and performance.

In [ ]:
drifted_test_probability = (
    selected_manual_pipeline
    .predict_proba(
        X_test_drifted
    )[:, 1]
)

drifted_test_metrics = (
    calculate_model_metrics(
        actual=y_test,
        probability=(
            drifted_test_probability
        ),
        threshold=(
            CLASSIFICATION_THRESHOLD
        ),
    )
)

performance_under_drift = (
    pd.DataFrame(
        [
            {
                "condition": (
                    "original_test"
                ),
                **test_metrics,
            },
            {
                "condition": (
                    "simulated_drift"
                ),
                **drifted_test_metrics,
            },
        ]
    )
)

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_churn",
    "recall_churn",
    "f1_churn",
    "roc_auc",
    "average_precision",
    "log_loss",
    "mean_probability",
    "predicted_churn_rate",
]

original_metric_row = (
    performance_under_drift.loc[
        performance_under_drift[
            "condition"
        ]
        == "original_test"
    ]
    .iloc[0]
)

drifted_metric_row = (
    performance_under_drift.loc[
        performance_under_drift[
            "condition"
        ]
        == "simulated_drift"
    ]
    .iloc[0]
)

performance_change_rows = []

for metric in metric_columns:
    performance_change_rows.append(
        {
            "metric": metric,
            "original_value": (
                original_metric_row[
                    metric
                ]
            ),
            "drifted_value": (
                drifted_metric_row[
                    metric
                ]
            ),
            "absolute_change": (
                drifted_metric_row[
                    metric
                ]
                - original_metric_row[
                    metric
                ]
            ),
        }
    )

performance_change = pd.DataFrame(
    performance_change_rows
)

performance_under_drift.to_csv(
    OUTPUT_DIR
    / "06_performance_under_drift.csv",
    index=False,
)

performance_change.to_csv(
    OUTPUT_DIR
    / "07_performance_change.csv",
    index=False,
)

display(
    performance_under_drift.style.format(
        {
            column: "{:.4f}"
            for column in (
                metric_columns
            )
        }
    )
)

display(
    performance_change.style.format(
        {
            "original_value": "{:.4f}",
            "drifted_value": "{:.4f}",
            "absolute_change": (
                "{:+.4f}"
            ),
        }
    )
)

## 12. Measure simulated prediction drift

Prediction PSI is calculated between the original and drifted test probabilities.

In [ ]:
simulated_prediction_psi = (
    numeric_psi(
        pd.Series(
            test_probability
        ),
        pd.Series(
            drifted_test_probability
        ),
    )
)

simulated_prediction_row = {
    "comparison": (
        "original_test_vs_"
        "simulated_drift"
    ),
    "model": "Random Forest",
    "probability_psi": (
        simulated_prediction_psi
    ),
    "prediction_drift_severity": (
        classify_psi(
            simulated_prediction_psi
        )
    ),
    "reference_mean_probability": (
        float(
            test_probability.mean()
        )
    ),
    "current_mean_probability": (
        float(
            drifted_test_probability.mean()
        )
    ),
    "mean_probability_change": (
        float(
            drifted_test_probability.mean()
            - test_probability.mean()
        )
    ),
    "reference_predicted_churn_rate": (
        test_metrics[
            "predicted_churn_rate"
        ]
    ),
    "current_predicted_churn_rate": (
        drifted_test_metrics[
            "predicted_churn_rate"
        ]
    ),
    "predicted_churn_rate_change": (
        drifted_test_metrics[
            "predicted_churn_rate"
        ]
        - test_metrics[
            "predicted_churn_rate"
        ]
    ),
}

prediction_drift_summary = pd.concat(
    [
        prediction_drift_summary,
        pd.DataFrame(
            [
                simulated_prediction_row
            ]
        ),
    ],
    ignore_index=True,
)

prediction_drift_summary.to_csv(
    OUTPUT_DIR
    / "08_prediction_drift_summary.csv",
    index=False,
)

display(
    prediction_drift_summary
)

## 13. Add the final AutoML test result as contextual evidence

The existing final prediction-evidence file is used only to summarise the original AutoML test performance.

No claim is made that AutoML performance under the synthetic feature changes was evaluated in this local notebook.

In [ ]:
performance_context_rows = [
    {
        "model": "Random Forest",
        "condition": "original_test",
        **test_metrics,
        "drifted_feature_scoring_available": (
            True
        ),
    },
    {
        "model": "Random Forest",
        "condition": "simulated_drift",
        **drifted_test_metrics,
        "drifted_feature_scoring_available": (
            True
        ),
    },
]

if (
    FINAL_PREDICTION_EVIDENCE_PATH
    .exists()
):
    final_prediction_evidence = (
        pd.read_csv(
            FINAL_PREDICTION_EVIDENCE_PATH
        )
    )

    required_auto_columns = {
        "actual_class",
        "automl_probability",
    }

    if required_auto_columns.issubset(
        final_prediction_evidence.columns
    ):
        automl_actual = (
            pd.to_numeric(
                final_prediction_evidence[
                    "actual_class"
                ],
                errors="raise",
            )
            .astype(int)
            .to_numpy()
        )

        automl_probability = (
            pd.to_numeric(
                final_prediction_evidence[
                    "automl_probability"
                ],
                errors="raise",
            )
            .to_numpy()
        )

        automl_test_metrics = (
            calculate_model_metrics(
                actual=automl_actual,
                probability=(
                    automl_probability
                ),
                threshold=(
                    CLASSIFICATION_THRESHOLD
                ),
            )
        )

        performance_context_rows.append(
            {
                "model": (
                    "Azure AutoML "
                    "selected model"
                ),
                "condition": (
                    "original_test"
                ),
                **automl_test_metrics,
                "drifted_feature_scoring_available": (
                    False
                ),
            }
        )

performance_context = pd.DataFrame(
    performance_context_rows
)

performance_context.to_csv(
    OUTPUT_DIR
    / "09_model_performance_context.csv",
    index=False,
)

display(
    performance_context
)

## 14. Generate operational drift alerts

The alert rules are deliberately explicit:

- feature PSI of 0.10 or above creates a warning;
- feature PSI of 0.25 or above creates a significant alert;
- probability PSI of 0.10 or above creates a warning;
- ROC-AUC, average precision or F1 deterioration of 0.03 or more creates a performance alert;
- recall deterioration of 0.05 or more creates a recall alert;
- a relative predicted-churn-rate change of 20% or more creates an output-volume alert.

In a production system, these thresholds should be reviewed against business risk tolerance and monitoring frequency.

In [ ]:
alert_rows = []

significant_features = (
    simulated_feature_drift.loc[
        simulated_feature_drift[
            "psi"
        ]
        >= 0.25
    ]
)

warning_features = (
    simulated_feature_drift.loc[
        (
            simulated_feature_drift[
                "psi"
            ]
            >= 0.10
        )
        & (
            simulated_feature_drift[
                "psi"
            ]
            < 0.25
        )
    ]
)

for _, row in (
    significant_features.iterrows()
):
    alert_rows.append(
        {
            "alert_type": (
                "feature_drift"
            ),
            "severity": (
                "significant"
            ),
            "item": row[
                "feature"
            ],
            "observed_value": (
                row["psi"]
            ),
            "threshold": 0.25,
            "recommended_action": (
                "Investigate the feature, "
                "confirm data quality and "
                "consider model retraining."
            ),
        }
    )

for _, row in (
    warning_features.iterrows()
):
    alert_rows.append(
        {
            "alert_type": (
                "feature_drift"
            ),
            "severity": "warning",
            "item": row[
                "feature"
            ],
            "observed_value": (
                row["psi"]
            ),
            "threshold": 0.10,
            "recommended_action": (
                "Continue monitoring and "
                "review the feature trend."
            ),
        }
    )

if simulated_prediction_psi >= 0.10:
    alert_rows.append(
        {
            "alert_type": (
                "prediction_drift"
            ),
            "severity": (
                "significant"
                if simulated_prediction_psi
                >= 0.25
                else "warning"
            ),
            "item": (
                "prediction_probability"
            ),
            "observed_value": (
                simulated_prediction_psi
            ),
            "threshold": 0.10,
            "recommended_action": (
                "Review prediction volumes, "
                "performance and model calibration."
            ),
        }
    )

monitored_performance_rules = {
    "roc_auc": 0.03,
    "average_precision": 0.03,
    "f1_churn": 0.03,
    "recall_churn": 0.05,
}

for metric, deterioration_limit in (
    monitored_performance_rules.items()
):
    original_value = float(
        original_metric_row[
            metric
        ]
    )

    drifted_value = float(
        drifted_metric_row[
            metric
        ]
    )

    deterioration = (
        original_value
        - drifted_value
    )

    if deterioration >= (
        deterioration_limit
    ):
        alert_rows.append(
            {
                "alert_type": (
                    "performance_deterioration"
                ),
                "severity": (
                    "significant"
                ),
                "item": metric,
                "observed_value": (
                    deterioration
                ),
                "threshold": (
                    deterioration_limit
                ),
                "recommended_action": (
                    "Investigate model decay "
                    "and evaluate retraining."
                ),
            }
        )

original_predicted_rate = float(
    test_metrics[
        "predicted_churn_rate"
    ]
)

drifted_predicted_rate = float(
    drifted_test_metrics[
        "predicted_churn_rate"
    ]
)

if original_predicted_rate > 0:
    relative_rate_change = (
        abs(
            drifted_predicted_rate
            - original_predicted_rate
        )
        / original_predicted_rate
    )
else:
    relative_rate_change = np.nan

if (
    not pd.isna(
        relative_rate_change
    )
    and relative_rate_change >= 0.20
):
    alert_rows.append(
        {
            "alert_type": (
                "prediction_volume"
            ),
            "severity": "warning",
            "item": (
                "predicted_churn_rate"
            ),
            "observed_value": (
                relative_rate_change
            ),
            "threshold": 0.20,
            "recommended_action": (
                "Review campaign capacity, "
                "threshold suitability and "
                "possible population change."
            ),
        }
    )

drift_alerts = pd.DataFrame(
    alert_rows,
    columns=[
        "alert_type",
        "severity",
        "item",
        "observed_value",
        "threshold",
        "recommended_action",
    ],
)

drift_alerts.to_csv(
    OUTPUT_DIR
    / "10_drift_alerts.csv",
    index=False,
)

display(
    drift_alerts
)

print(
    "Number of alerts:",
    len(drift_alerts),
)

## 15. Create the feature-drift chart

The chart displays the features with the highest simulated PSI values. Horizontal reference lines show the warning and significant-drift thresholds.

In [ ]:
top_feature_drift = (
    simulated_feature_drift
    .sort_values(
        "psi",
        ascending=False,
    )
    .head(15)
    .sort_values(
        "psi",
        ascending=True,
    )
)

figure, axis = plt.subplots(
    figsize=(10, 7)
)

axis.barh(
    top_feature_drift[
        "feature"
    ],
    top_feature_drift[
        "psi"
    ],
)

axis.axvline(
    0.10,
    linestyle="--",
    label="Warning threshold (0.10)",
)

axis.axvline(
    0.25,
    linestyle=":",
    label=(
        "Significant threshold (0.25)"
    ),
)

axis.set_title(
    "Top simulated feature-drift PSI values"
)

axis.set_xlabel(
    "Population Stability Index"
)

axis.set_ylabel(
    "Feature"
)

axis.legend()

axis.grid(
    axis="x",
    alpha=0.3,
)

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "11_feature_drift_chart.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 16. Create the prediction-drift chart

The chart compares the original and drifted test probability distributions.

In [ ]:
figure, axis = plt.subplots(
    figsize=(10, 6)
)

common_bins = np.linspace(
    0,
    1,
    31,
)

axis.hist(
    test_probability,
    bins=common_bins,
    density=True,
    alpha=0.55,
    label="Original test",
)

axis.hist(
    drifted_test_probability,
    bins=common_bins,
    density=True,
    alpha=0.55,
    label="Simulated drift",
)

axis.set_title(
    "Prediction probability distribution "
    "before and after simulated drift"
)

axis.set_xlabel(
    "Predicted churn probability"
)

axis.set_ylabel(
    "Density"
)

axis.legend()

axis.grid(
    alpha=0.3,
)

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "12_prediction_drift_chart.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 17. Create the performance-deterioration chart

This figure focuses on the principal discrimination and classification metrics.

In [ ]:
chart_metrics = [
    "roc_auc",
    "average_precision",
    "f1_churn",
    "recall_churn",
    "balanced_accuracy",
]

chart_data = (
    performance_under_drift[
        [
            "condition",
            *chart_metrics,
        ]
    ]
    .set_index(
        "condition"
    )
    .T
)

axis = chart_data.plot(
    kind="bar",
    figsize=(10, 6),
)

axis.set_title(
    "Model performance before and "
    "after simulated drift"
)

axis.set_xlabel(
    "Metric"
)

axis.set_ylabel(
    "Score"
)

axis.set_ylim(
    0,
    1,
)

axis.tick_params(
    axis="x",
    rotation=20,
)

axis.grid(
    axis="y",
    alpha=0.3,
)

figure = axis.get_figure()

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "13_performance_under_drift_chart.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 18. Produce a dissertation-ready monitoring conclusion

The conclusion reports:

- the most drifted observed features;
- the most drifted simulated features;
- prediction-distribution change;
- model-performance change;
- the number of triggered alerts;
- the main methodological limitation.

In [ ]:
observed_top_features = (
    feature_drift_summary
    .sort_values(
        "psi",
        ascending=False,
    )
    .head(5)
)

simulated_top_features = (
    simulated_feature_drift
    .sort_values(
        "psi",
        ascending=False,
    )
    .head(5)
)

conclusion_lines = [
    "MODEL DRIFT MONITORING CONCLUSION",
    "",
    (
        "Observed validation-to-test "
        "prediction PSI: "
        f"{observed_prediction_psi:.4f} "
        f"({classify_psi(observed_prediction_psi)})."
    ),
    (
        "Simulated prediction PSI: "
        f"{simulated_prediction_psi:.4f} "
        f"({classify_psi(simulated_prediction_psi)})."
    ),
    (
        "Original test ROC-AUC: "
        f"{test_metrics['roc_auc']:.4f}."
    ),
    (
        "Drifted test ROC-AUC: "
        f"{drifted_test_metrics['roc_auc']:.4f}."
    ),
    (
        "Original test average precision: "
        f"{test_metrics['average_precision']:.4f}."
    ),
    (
        "Drifted test average precision: "
        f"{drifted_test_metrics['average_precision']:.4f}."
    ),
    (
        "Original test churn F1: "
        f"{test_metrics['f1_churn']:.4f}."
    ),
    (
        "Drifted test churn F1: "
        f"{drifted_test_metrics['f1_churn']:.4f}."
    ),
    (
        "Operational alerts triggered: "
        f"{len(drift_alerts)}."
    ),
    "",
    "Highest observed PSI features:",
]

for _, row in (
    observed_top_features.iterrows()
):
    conclusion_lines.append(
        (
            f"- {row['comparison']} / "
            f"{row['feature']}: "
            f"PSI={row['psi']:.4f} "
            f"({row['drift_severity']})"
        )
    )

conclusion_lines.extend(
    [
        "",
        "Highest simulated PSI features:",
    ]
)

for _, row in (
    simulated_top_features.iterrows()
):
    conclusion_lines.append(
        (
            f"- {row['feature']}: "
            f"PSI={row['psi']:.4f} "
            f"({row['drift_severity']})"
        )
    )

conclusion_lines.extend(
    [
        "",
        (
            "The stress test demonstrates how "
            "feature and prediction monitoring can "
            "identify changes before performance "
            "deterioration becomes operationally "
            "significant."
        ),
        (
            "The simulated scenario is illustrative "
            "and should not be interpreted as a "
            "forecast of future customer behaviour."
        ),
        (
            "The locally executable Random Forest "
            "pipeline was rescored under drift. "
            "The AutoML model should be monitored "
            "using the same framework after endpoint "
            "deployment or through an isolated Azure "
            "scoring environment."
        ),
    ]
)

conclusion_text = "\n".join(
    conclusion_lines
)

print(
    conclusion_text
)

(
    OUTPUT_DIR
    / "14_drift_monitoring_conclusion.txt"
).write_text(
    conclusion_text,
    encoding="utf-8",
)

## 19. Save reproducibility metadata and verify outputs

The metadata file records the random seed, model threshold, drift thresholds, input paths and simulated feature changes.

In [ ]:
metadata = {
    "random_state": RANDOM_STATE,
    "target_column": TARGET_COLUMN,
    "classification_threshold": (
        CLASSIFICATION_THRESHOLD
    ),
    "psi_warning_threshold": 0.10,
    "psi_significant_threshold": (
        0.25
    ),
    "training_path": str(
        TRAIN_PATH
    ),
    "validation_path": str(
        VALIDATION_PATH
    ),
    "test_path": str(
        TEST_PATH
    ),
    "manual_model_path": str(
        MANUAL_MODEL_PATH
    ),
    "final_prediction_evidence_path": (
        str(
            FINAL_PREDICTION_EVIDENCE_PATH
        )
    ),
    "training_rows": int(
        len(X_train)
    ),
    "validation_rows": int(
        len(X_validation)
    ),
    "test_rows": int(
        len(X_test)
    ),
    "predictor_count": int(
        X_train.shape[1]
    ),
    "simulated_features": (
        applied_drift_summary[
            "feature"
        ].tolist()
    ),
}

(
    OUTPUT_DIR
    / "15_drift_run_metadata.json"
).write_text(
    json.dumps(
        metadata,
        indent=2,
    ),
    encoding="utf-8",
)


expected_outputs = [
    "01_feature_drift_summary.csv",
    "02_numeric_psi_results.csv",
    "03_categorical_drift_results.csv",
    "04_simulated_drift_definition.csv",
    "05_simulated_feature_drift.csv",
    "06_performance_under_drift.csv",
    "07_performance_change.csv",
    "08_prediction_drift_summary.csv",
    "09_model_performance_context.csv",
    "10_drift_alerts.csv",
    "11_feature_drift_chart.png",
    "12_prediction_drift_chart.png",
    "13_performance_under_drift_chart.png",
    "14_drift_monitoring_conclusion.txt",
    "15_drift_run_metadata.json",
]

output_verification = pd.DataFrame(
    {
        "file": expected_outputs,
        "exists": [
            (
                OUTPUT_DIR
                / file_name
            ).exists()
            for file_name in (
                expected_outputs
            )
        ],
    }
)

display(
    output_verification
)

if output_verification[
    "exists"
].all():
    print(
        "\nMODEL DRIFT MONITORING "
        "COMPLETED SUCCESSFULLY"
    )

    print(
        "Output folder:",
        OUTPUT_DIR.resolve(),
    )

else:
    missing_outputs = (
        output_verification.loc[
            ~output_verification[
                "exists"
            ],
            "file",
        ]
        .tolist()
    )

    raise FileNotFoundError(
        "Missing expected outputs: "
        + ", ".join(
            missing_outputs
        )
    )